# Feature — Tool Library Showcase

Builtin tools under agenttune.agentic.tools.builtin, exercised for real: SQL (local SQLite), code execution (RunPythonTool/RunBashTool), file I/O (Read/Write/ListDir), and search (GrepTool). GitHubTool/SlackTool/PlaywrightTool exist in the library too but aren't exercised here — no GitHub/Slack credentials or the playwright package/browser binaries are available in this environment.


In [1]:
"""
Feature — Tool Library showcase.
=================================

Exercises the builtin tools under `agenttune.agentic.tools.builtin` that run fully
locally (no external API key / network / browser binary required in this
environment): SQL (SQLDatabaseTool over a local SQLite file), code execution
(RunPythonTool, RunBashTool), file I/O (ReadFileTool, WriteFileTool, ListDirTool),
and search (GrepTool). Each is called for real and its real ToolResult is printed.

Three more builtin tools exist in the same library (GitHubTool, SlackTool,
PlaywrightTool) but are NOT exercised here: GitHubTool/SlackTool need real
API credentials (GITHUB/SLACK tokens) not present in this environment, and
PlaywrightTool needs the `playwright` package + browser binaries, which are
not installed here. That's an environment limitation, not a code gap — the
tools exist and are unit-tested elsewhere in the repo's test suite.

Finally, a couple of these tools are wired into a real `Project` + `DictToolHarness`
agent episode, tying the tool library back to the same agentic spine used
elsewhere in these notebooks.
"""
import os
import sqlite3
import tempfile

In [2]:
from agenttune.agentic.tools.builtin.sql import SQLDatabaseTool
from agenttune.agentic.tools.builtin.code_tools import RunPythonTool, RunBashTool
from agenttune.agentic.tools.builtin.file_tools import ReadFileTool, WriteFileTool, ListDirTool
from agenttune.agentic.tools.builtin.search_tools import GrepTool

In [3]:
from agenttune.agentic import Project, DictToolHarness, ReActStrategy

In [4]:
def main():
    workdir = tempfile.mkdtemp(prefix="agenttune_tool_showcase_")
    print(f"[workdir] {workdir}")

    # 1) SQL — local SQLite database, real query
    db_path = os.path.join(workdir, "demo.db")
    conn = sqlite3.connect(db_path)
    conn.execute("CREATE TABLE accounts (id INTEGER, name TEXT, balance REAL)")
    conn.executemany(
        "INSERT INTO accounts VALUES (?, ?, ?)",
        [(1, "acme_corp", 15000.50), (2, "globex", 2300.00)],
    )
    conn.commit()
    conn.close()

    sql_tool = SQLDatabaseTool(f"sqlite:///{db_path}")
    result = sql_tool.execute(action="sql_db_query", input="SELECT name, balance FROM accounts ORDER BY balance DESC")
    print(f"[sql]     query result -> {result.output}")

    # 2) Code execution — real Python subprocess
    py_tool = RunPythonTool()
    result = py_tool.execute(code="print(sum(range(1, 101)))")
    print(f"[python]  sum(1..100) -> {result.output.strip()!r}  success={result.success}")

    bash_tool = RunBashTool()
    result = bash_tool.execute(command="echo hello from a real subprocess")
    print(f"[bash]    -> {result.output.strip()!r}  success={result.success}")

    # 3) File I/O — real filesystem
    target = os.path.join(workdir, "notes.txt")
    write_tool = WriteFileTool()
    result = write_tool.execute(path=target, content="AgentTune tool library demo\nsecond line\n")
    print(f"[write]   -> {result.output}")

    read_tool = ReadFileTool()
    result = read_tool.execute(path=target)
    print(f"[read]    -> {result.output!r}")

    list_tool = ListDirTool()
    result = list_tool.execute(path=workdir)
    print(f"[list_dir] -> {result.output}")

    # 4) Search — real grep subprocess
    grep_tool = GrepTool()
    result = grep_tool.execute(pattern="second", path=target)
    print(f"[grep]    -> {result.output.strip()!r}")

    # 5) Tie back to the spine: a tiny real agent episode using two of these tools
    tools = {
        "list_dir": lambda path=".": list_tool.execute(path=path).output,
        "read_file": lambda path: read_tool.execute(path=path).output,
    }

    def file_inspector_policy(state):
        if state.step == 0:
            return {"name": "list_dir", "arguments": {"path": workdir}}
        if state.step == 1:
            return {"name": "read_file", "arguments": {"path": target}}
        return {"name": "finish", "arguments": {"answer": "inspected workdir + read notes.txt"}}

    proj = Project(strategy=ReActStrategy(file_inspector_policy, max_steps=4),
                    harness=DictToolHarness(tools, max_steps=4))
    log = proj.infer("inspect the workdir and read notes.txt")
    tool_calls = [e.payload["action"]["name"] for e in log.events if e.kind.name == "TOOL_CALL"]
    print(f"[agent]   real episode over the tool library -> tool calls: {tool_calls}")

    print("\nNOT exercised here (need credentials/packages not present in this "
          "environment): GitHubTool (GITHUB token), SlackTool (SLACK token), "
          "PlaywrightTool (playwright package + browser binaries).")

In [5]:
if __name__ == "__main__":
    main()

[workdir] /tmp/agenttune_tool_showcase_hv944o8z
[sql]     query result -> [('acme_corp', 15000.5), ('globex', 2300.0)]
[python]  sum(1..100) -> '5050'  success=True
[bash]    -> 'hello from a real subprocess'  success=True
[write]   -> Written to /tmp/agenttune_tool_showcase_hv944o8z/notes.txt
[read]    -> 'AgentTune tool library demo\nsecond line\n'
[list_dir] -> ['demo.db', 'notes.txt']
[grep]    -> 'second line'
[agent]   real episode over the tool library -> tool calls: ['list_dir', 'read_file', 'finish']

NOT exercised here (need credentials/packages not present in this environment): GitHubTool (GITHUB token), SlackTool (SLACK token), PlaywrightTool (playwright package + browser binaries).
